# 먼저 생성하고, 필요하면 다시 역확산해서 획 보정

1~3번: 모델/자료 준비와 네 문자 선택. 최초 생성은 INIT_MODE=noise만 허용합니다.
4번: 획 유도 없이 최초 이미지를 생성하고 저장합니다. 이 결과를 먼저 확인하세요.
이미 저장한 생성물을 쓰려면 FIRST_IMAGE에 같은 크기의 2×2 이미지 경로를 넣습니다.
5번: 최초 생성물에 작은 잡음을 다시 넣고 동일 잠재값으로 두 경로를 비교합니다.
6번: 처음 생성 / 재역확산만 / 획 구조 유도 재역확산을 비교하고 진단 ZIP을 받습니다.

재보정은 깨끗한 정답 픽셀을 사용하지 않습니다. 표준 글자는 획 추출·구조 특성의 조건입니다.
첫 생성물과의 RGB 오차는 원래 모양 보존용이며, 구조 손실은 기존 정규화 획 특성을 사용합니다.
이 구조 지표의 추가 획 탐지 한계는 여전히 남아 있으므로 개선을 보장하지 않습니다.
최초 생성은 프롬프트와 잡음만 사용합니다. 폰트 입력 기본값은 폐기했습니다.
오류 판정의 검증된 임계값은 없어서 4번 결과를 보고 사용자가 5번 실행 여부를 정합니다.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile, json
root = Path(tempfile.mkdtemp(prefix='hsqm_diffusion_score_', dir='/content'))
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
    'https://github.com/yechan25/hsqm.git', str(root)], check=True)
commit = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.35.1', 'transformers==4.49.0', 'accelerate', 'safetensors',
    'timm', 'pandas', 'numpy', 'tqdm', 'matplotlib', 'pillow', 'scipy',
    'scikit-image', 'pyyaml', 'fonttools', 'ipywidgets'], check=True)
sys.path.insert(0, str(root))
for key in list(sys.modules):
    if key.split('.')[0] in ('stroke_model', 'font_test', 'diffusion_model'):
        del sys.modules[key]
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab에서 GPU 런타임으로 바꿔주세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('실행 GitHub 커밋:', commit)
from google.colab import drive, output
drive.mount('/content/drive')
output.enable_custom_widget_manager()
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','fonts-nanum'],check=True)

In [ ]:
from stroke_model.convert_reference_csv import existing_path
from diffusion_model.score_pilot import training_catalog, generate_pilot, score_pilot, load_catalog_reference
from diffusion_model.pilot_data import prepare_pilot_data
from diffusion_model.multiglyph_pilot import generate_multiglyph_pilot
from IPython.display import display
import ipywidgets as widgets
HSQM = existing_path('/content/drive/Shareddrives/2026 자율연구/HSQM')
CHECKPOINT = None  # 폰트 실험에서 사용한 best.pt의 전체 경로를 넣으면 그 모델 사용
DATA_ROOT = HSQM/'dataset/automated_train_dataset'  # 기존 학습 ZIP/TAR가 있는 폴더
CSV_PATH = None  # None: 체크포인트 옆에 저장된 학습 CSV를 자동 사용
TRAIN_MANIFEST = None  # 기존 세션의 유효한 목록이 있을 때만 선택적으로 지정. 없어도 자동 재생성
REFERENCE_ONLY = True  # 시험에는 기준 글자/기준 획만 사용. 전체 학습 NPZ 변환 생략
SCORE_CONFIG = {}  # 폰트 시험에서 값을 바꿨다면 동일한 설정 입력
SOURCE_CONFIG = {}
GLYPH_COUNTS = [4]
CANVAS_SIZE = 576
FIXED_CELL_PIXELS = None  # None: 전체 크기 고정. 192: 글자당 크기 고정 비교
STRENGTHS = [.65]
SEEDS = [100]
STEPS = 100
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
if CHECKPOINT is None:
    candidates = sorted((HSQM/'v2_topology_runs').glob('*/best.pt'))
    if not candidates:
        raise FileNotFoundError('CHECKPOINT에 기존 Topo best.pt를 지정하세요.')
    CHECKPOINT = candidates[-1]
CHECKPOINT = existing_path(CHECKPOINT)
print('사용할 획 모델:', CHECKPOINT, flush=True)
preparation = prepare_pilot_data(CHECKPOINT, DATA_ROOT, csv_path=CSV_PATH, manifest=TRAIN_MANIFEST,
    reference_only=REFERENCE_ONLY)
TRAIN_MANIFEST = Path(preparation['manifest'])
print('데이터 준비 완료. 3번 셀에서 기준 글자를 선택하세요.', flush=True)

# Separate experimental path; old pixel-control notebook is preserved.
SEED=100  # Repeat with 101, 102, ...; keep all results
INIT_MODE='noise'  # First generation must not use an input glyph image
STRENGTH=.65
AUTO_FONT_INPUT=False
INPUT_SHEET=None

FIRST_IMAGE=None  # Optional existing generated 2x2 image path
REPAIR_STEPS=100
REPAIR_STRENGTH=.2
REPAIR_SEED=101
PRESERVATION_WEIGHT=1.
REPAIR_GUIDANCE=dict(local_edit=False,clean_step_ratio=.003,start_fraction=.25,end_fraction=1.,
    every=1,backtracks=6,state_update='clean_residual',max_new_local_ink=None,
    max_state_change_to_base_step_ratio=.25)

In [ ]:
import io, math
import numpy as np
from PIL import Image, ImageDraw
catalog = training_catalog(TRAIN_MANIFEST)
by_row = {item['manifest_row']: item for item in catalog}
needed = max(GLYPH_COUNTS)
if len(catalog) < needed:
    raise ValueError(f'사용 가능한 학습 기준은 {len(catalog)}개입니다. GLYPH_COUNTS를 줄이세요.')
reference_select = widgets.SelectMultiple(
    options=[(f"train row {item['manifest_row']} | {item['char_id'] or '(그림 확인)'}", item['manifest_row']) for item in catalog],
    value=tuple(item['manifest_row'] for item in catalog[:needed]), rows=12,
    description='여러 글자:', layout=widgets.Layout(width='98%'))
preview = widgets.Image(format='png', width=500, height=500)
def update_preview(change=None):
    selected = list(reference_select.value)[:needed]
    side = math.ceil(math.sqrt(needed))
    sheet = Image.new('RGB', (160*side, 180*side), 'white')
    draw = ImageDraw.Draw(sheet)
    for index, row in enumerate(selected):
        reference,_ = load_catalog_reference(by_row[row])
        glyph = Image.fromarray(np.uint8((1-reference[0])*255)).convert('RGB')
        x, y = (index%side)*160, (index//side)*180
        sheet.paste(glyph.resize((160,160)), (x,y))
        draw.text((x+4,y+160), f'slot {index+1} | row {row}', fill='black')
    buffer=io.BytesIO(); sheet.save(buffer,format='PNG'); preview.value=buffer.getvalue()
reference_select.observe(update_preview,names='value'); update_preview()
display(widgets.VBox([reference_select,preview]))
print('정확히 4개를 선택하세요. 같은 4개로 기본/유도 경로를 비교합니다.')
print('문자명이 없는 경우 그림으로 확인하세요. 서로 다른 글자를 비교하려면 같은 글자의 폰트 변형을 중복 선택하지 마세요.')
import ipywidgets as widgets
glyph_box=widgets.Text(description='4문자:',placeholder='예: 가 걔 곁 괴 (선택 순서와 일치)')
font_paths=sorted(Path('/usr/share/fonts/truetype/nanum').glob('*.ttf'))
default_font=next((str(p) for p in font_paths if p.name=='NanumMyeongjo.ttf'),None)
font_box=widgets.Dropdown(options=[(p.name,str(p)) for p in font_paths],value=default_font,description='입력 폰트:')
prompt_box=widgets.Textarea(description='Prompt',value='Four large Korean Hangul characters in a 2 by 2 layout. Preserve each input character and its position. Black ink on plain white background. No additional text, tables, grid lines, captions or logos.',layout=widgets.Layout(width='95%',height='90px'))
if INIT_MODE=='external' and AUTO_FONT_INPUT:
    display(glyph_box,font_box,prompt_box)
else:
    display(glyph_box,prompt_box)

In [ ]:
from datetime import datetime
from huggingface_hub import model_info
from diffusion_model.two_stage_repair import generate_first_stage
if not prompt_box.value.strip():raise ValueError('선택한 4문자와 순서가 일치하는 프롬프트를 입력하세요.')
RUN_DIR=HSQM/'two_stage_repair'/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True,exist_ok=False)
MODEL_REVISION=model_info(MODEL_ID).sha
OUTPUT_DIR=RUN_DIR/'stage1'
if FIRST_IMAGE is None:
    input_metadata=None
    run_input_sheet=INPUT_SHEET
    if INIT_MODE=='external' and AUTO_FONT_INPUT and INPUT_SHEET is None:
        from diffusion_model.crossfont_input import compose_font_input
        from diffusion_model.score_pilot import training_catalog
        catalog={r['manifest_row']:r for r in training_catalog(TRAIN_MANIFEST)}
        glyphs=glyph_box.value.split()
        if len(glyphs)==1 and len(glyphs[0])==4:glyphs=list(glyphs[0])
        if not font_box.value:raise ValueError('입력 폰트를 선택하세요.')
        image,input_metadata=compose_font_input([catalog[r] for r in reference_select.value],glyphs,font_box.value,CANVAS_SIZE)
        import tempfile
        input_path=Path(tempfile.mkdtemp(prefix='hsqm_crossfont_',dir='/content'))/'input.png'
        image.save(input_path)
        run_input_sheet=input_path
        print('독립 폰트 입력 (선택한 표준 문자/순서와 맞는지 확인):');display(image)
    first=generate_first_stage(TRAIN_MANIFEST,list(reference_select.value),CHECKPOINT,OUTPUT_DIR,
        model_id=MODEL_ID,revision=MODEL_REVISION,steps=STEPS,canvas_size=CANVAS_SIZE,
        seed=SEED,strength=STRENGTH,init_mode=INIT_MODE,input_sheet=run_input_sheet,
        prompt_text=prompt_box.value.strip())
    FIRST_RESULT=Path(first['generated_image'])
    if input_metadata:
        (OUTPUT_DIR/'input_font.json').write_text(json.dumps(input_metadata,ensure_ascii=False,indent=2),encoding='utf-8')
else:
    FIRST_RESULT=Path(FIRST_IMAGE)
    if not FIRST_RESULT.is_file():raise FileNotFoundError(FIRST_RESULT)
(RUN_DIR/'git_commit.txt').write_text(commit,encoding='utf-8')
print('최초 생성 결과 — 보정이 필요하면 5번 셀을 실행하세요:',FIRST_RESULT)
display(Image.open(FIRST_RESULT))

In [ ]:
from diffusion_model.two_stage_repair import repair_generated_image
OUTPUT_DIR=RUN_DIR/('stage2_'+datetime.now().strftime('%H%M%S_%f'))
generation=repair_generated_image(TRAIN_MANIFEST,list(reference_select.value),CHECKPOINT,
    FIRST_RESULT,OUTPUT_DIR,model_id=MODEL_ID,revision=MODEL_REVISION,
    canvas_size=CANVAS_SIZE,steps=REPAIR_STEPS,strength=REPAIR_STRENGTH,seed=REPAIR_SEED,
    preservation_weight=PRESERVATION_WEIGHT,guidance=REPAIR_GUIDANCE,
    prompt_text=prompt_box.value.strip())
print('두 재보정 경로의 동일 시작 잠재값:',generation['initial_latent_sha256'])
print('최초 생성물 보존 해시:',generation['first_generated_sha256'])
print('구조 손실 및 보존 오차:',generation['structure_audit'])

In [ ]:
import matplotlib.pyplot as plt
pages={p['method']:p for p in generation['pages']}
items=[('First generation',FIRST_RESULT),('Re-denoise only',pages['baseline']['input']),
       ('Stroke-guided re-denoise',pages['guided']['input'])]
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,(title,path) in zip(axes,items):
    with Image.open(path) as image:ax.imshow(image.convert('RGB'))
    ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(RUN_DIR/'two_stage_comparison.png',dpi=140);plt.show()
import zipfile
from google.colab import files
archive=RUN_DIR.parent/(RUN_DIR.name+'_two_stage_debug.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in RUN_DIR.rglob('*'):
        if path.is_file():z.write(path,path.relative_to(RUN_DIR))
    z.write(FIRST_RESULT,'first_generated_input.png')
files.download(str(archive))